# QLoRA fine-tuning on a Kaggle T4: when it helps, when it hurts, how to serve it

Question: when does fine-tuning beat prompting, how do overfitting and catastrophic forgetting look in real numbers, and how does a LoRA adapter run in a vLLM server?

Setup: two Tesla T4 GPUs (16 GB each, Turing, no bf16, so fp16 everywhere), `Qwen/Qwen2.5-0.5B-Instruct`, the `ag_news` dataset. Training uses GPU 0. The vLLM server uses GPU 1. Task: read a news text and answer with strict JSON, `{"label": "<World|Sports|Business|Sci/Tech>"}`.

Terms used below:

- **Prompt engineering**: you change the input text (instructions, examples) and leave the model weights alone.
- **Fine-tuning**: you change the model weights with new training examples.
- **SFT** (supervised fine-tuning): fine-tuning on pairs of input and wanted output. The loss is the next-token error on the wanted output.
- **PEFT** (parameter-efficient fine-tuning): a family of methods that train only a small number of extra weights and freeze the rest.
- **LoRA** (low-rank adaptation): a PEFT method. For a frozen weight matrix W, it learns two small matrices A and B. The layer then uses W + (alpha / r) x B A. The rank `r` sets the size of A and B.
- **Adapter**: the saved small file that holds the LoRA matrices. It cannot run alone. It needs the base model.
- **QLoRA**: LoRA on top of a base model stored in 4 bits. The base is frozen and quantized. Only the LoRA matrices train, in higher precision. This cuts memory.
- **NF4** (4-bit NormalFloat): the 4-bit number format that QLoRA uses for the base weights.
- **Overfitting**: the model learns the training examples, not the task. Training loss falls. Validation loss stops falling and then rises.
- **Catastrophic forgetting**: after fine-tuning on one task, the model gets worse at other tasks it did well before.
- **Merging**: you add the LoRA change into the base weights once (`merge_and_unload`). The result is one normal model with no adapter.
- **Valid-JSON rate**: the share of replies that are exactly one JSON object with a known label, with no extra text.
- **Accuracy**: the share of replies whose label (found even inside extra text) equals the true label.

## Predict before running

Write your own guess first. Then compare it with the tables at the end.

| Step | What you measure | Expected direction |
|---|---|---|
| Zero-shot vs 4-shot prompt | Accuracy and valid-JSON rate of the base model | Examples in the prompt should help the format. Check if they also help accuracy. |
| QLoRA SFT | Same two numbers on the test set | Both should be higher than with prompting. The model learns the exact output format. |
| 64 examples, 8 epochs | Train loss and validation loss | Train loss keeps falling. Validation loss reaches a minimum and then rises. |
| Forgetting probe | Math accuracy of base vs base plus adapter | The adapter may lower it. The sample is small, so read the result with care. |
| vLLM adapter serving | Accuracy and median latency of `base` vs `news` | `base` stays poor. `news` matches the fine-tuned model. `news` costs a little extra latency. |
| Merged model | Same numbers as the adapter | Accuracy is the same within noise. Latency is equal or lower, because there is no adapter step. |

In [ ]:
# 1. Install the libraries (into the Kaggle VM, not your laptop)
# If pip changes the torch version, restart the kernel once and skip this cell.
!pip install -q peft bitsandbytes accelerate vllm 2>&1 | tail -3
!nvidia-smi --query-gpu=index,name,memory.total,compute_cap --format=csv

In [ ]:
# 2. Helpers: pure Python, no GPU. Labels, prompts, JSON parser, metrics, balanced split, math answer parser.
import json, os, random, re, statistics, subprocess, time, urllib.request
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # this kernel sees GPU 0 only. vLLM gets GPU 1 later.
SEED = 42
LABELS = ["World", "Sports", "Business", "Sci/Tech"]
SYSTEM = ('Classify the news article into one topic: World, Sports, Business, or Sci/Tech. '
          'Reply with only this JSON and nothing else: {"label": "<topic>"}')
T0 = time.time()
def lap(name): print(f"[{(time.time()-T0)/60:5.1f} min] {name}")

def label_json(label):
    return json.dumps({"label": label})

def clip(text, n=600):
    return " ".join(text.split())[:n]

def build_messages(text, shots=()):
    # shots: list of (text, label) pairs. They become earlier user and assistant turns.
    msgs = [{"role": "system", "content": SYSTEM}]
    for t, l in shots:
        msgs += [{"role": "user", "content": "Article: " + clip(t)}, {"role": "assistant", "content": label_json(l)}]
    msgs.append({"role": "user", "content": "Article: " + clip(text)})
    return msgs

def pick_shots(rows, seed):
    # one training example per label, in a seeded random order
    first = {}
    for t, l in rows:
        first.setdefault(l, (t, l))
    shots = [first[l] for l in LABELS]
    random.Random(seed).shuffle(shots)
    return shots

def parse_label(text):
    # Returns (label or None, strict). strict is True only if the whole reply is one JSON object with a known label.
    t = text.strip()
    try:
        obj = json.loads(t)
        if isinstance(obj, dict) and obj.get("label") in LABELS:
            return obj["label"], True
    except ValueError:
        pass
    m = re.search(r"\{[^{}]*\}", t)
    if m:
        try:
            obj = json.loads(m.group(0))
            if isinstance(obj, dict) and obj.get("label") in LABELS:
                return obj["label"], False
        except ValueError:
            pass
    return None, False

def score(outputs, golds):
    parsed = [parse_label(o) for o in outputs]
    n = len(golds)
    return {"accuracy": sum(p[0] == g for p, g in zip(parsed, golds)) / n,
            "valid_json": sum(p[1] for p in parsed) / n, "n": n}

def balanced_split(labels, sizes, seed):
    # Returns one list of indices per size. Each list has the same number of items per class. Lists do not overlap.
    classes = sorted(set(labels))
    assert all(s % len(classes) == 0 for s in sizes), "each size must divide by the number of classes"
    rng = random.Random(seed)
    pools = {c: [i for i, l in enumerate(labels) if l == c] for c in classes}
    for p in pools.values():
        rng.shuffle(p)
    out, start = [], 0
    for s in sizes:
        per = s // len(classes)
        idx = []
        for c in classes:
            assert start + per <= len(pools[c]), "not enough items in class " + str(c)
            idx += pools[c][start:start + per]
        rng.shuffle(idx)
        out.append(idx)
        start += per
    return out

def mask_labels(prompt_ids, answer_ids):
    # Loss only on the answer tokens. -100 tells the loss function to ignore a position.
    return {"input_ids": prompt_ids + answer_ids, "labels": [-100] * len(prompt_ids) + answer_ids}

def pad_batch(features, pad_id):
    n = max(len(f["input_ids"]) for f in features)
    out = {"input_ids": [], "attention_mask": [], "labels": []}
    for f in features:
        k = n - len(f["input_ids"])
        out["input_ids"].append(f["input_ids"] + [pad_id] * k)
        out["attention_mask"].append([1] * len(f["input_ids"]) + [0] * k)
        out["labels"].append(f["labels"] + [-100] * k)
    return out

def split_log(log):
    # Trainer log_history -> ([(step, train_loss)], [(step, validation_loss)])
    tr = [(e["step"], e["loss"]) for e in log if "loss" in e and "eval_loss" not in e]
    ev = [(e["step"], e["eval_loss"]) for e in log if "eval_loss" in e]
    return tr, ev

def parse_gsm8k_answer(text):
    # Final number of a math answer. Order: '#### n', then \boxed{n}, then the last number in the text.
    m = re.search(r"####\s*\$?\s*(-?[\d,]*\.?\d+)", text) or re.search(r"\\boxed\{\s*\$?\s*(-?[\d,]*\.?\d+)", text)
    if m:
        s = m.group(1)
    else:
        nums = re.findall(r"-?\d[\d,]*\.?\d*", text)
        if not nums:
            return None
        s = nums[-1]
    try:
        return float(s.replace(",", "").rstrip("."))
    except ValueError:
        return None

def gsm8k_gold(answer_field):
    return float(answer_field.split("####")[-1].strip().replace(",", ""))

def gsm8k_correct(text, gold):
    p = parse_gsm8k_answer(text)
    return p is not None and abs(p - gold) < 1e-6

RESULTS = []   # rows for the summary table
def add(experiment, setting, accuracy=None, valid_json=None, latency_ms=None, n=None, note=""):
    RESULTS.append({"experiment": experiment, "setting": setting, "accuracy": accuracy, "valid_json": valid_json,
                    "median_latency_ms": latency_ms, "n": n, "note": note})

In [ ]:
# 3. Data: balanced samples from ag_news. Train 2,000, validation 300, test 300. Fixed seed.
import torch, transformers, datasets, peft, bitsandbytes, accelerate
import numpy as np, pandas as pd
from collections import Counter
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, Trainer, TrainingArguments, set_seed
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel
for m in (torch, transformers, datasets, peft, bitsandbytes, accelerate):
    print(m.__name__, m.__version__)
print("GPUs visible to this kernel:", torch.cuda.device_count(), "|", torch.cuda.get_device_name(0))

MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
tok = AutoTokenizer.from_pretrained(MODEL)
try:
    ds = load_dataset("ag_news")
except Exception as e:
    print("ag_news failed:", repr(e)[:200], "-> trying fancyzhx/ag_news")
    ds = load_dataset("fancyzhx/ag_news")
assert ds["train"].features["label"].names == LABELS, ds["train"].features["label"].names

tr_text, tr_y = ds["train"]["text"], [LABELS[i] for i in ds["train"]["label"]]
te_text, te_y = ds["test"]["text"], [LABELS[i] for i in ds["test"]["label"]]
tr_idx, va_idx = balanced_split(tr_y, [2000, 300], SEED)
(te_idx,) = balanced_split(te_y, [300], SEED)
train_rows = [(tr_text[i], tr_y[i]) for i in tr_idx]
val_rows = [(tr_text[i], tr_y[i]) for i in va_idx]
test_rows = [(te_text[i], te_y[i]) for i in te_idx]
(small_idx,) = balanced_split([y for _, y in train_rows], [64], SEED)
small_rows = [train_rows[i] for i in small_idx]
for name, rows in [("train", train_rows), ("validation", val_rows), ("test", test_rows), ("train-64", small_rows)]:
    print(f"{name:11s} {len(rows):5d}", dict(Counter(y for _, y in rows)))
SHOTS = pick_shots(train_rows, SEED)
print("few-shot labels, in prompt order:", [l for _, l in SHOTS])

def encode(text, label):
    prompt = tok.apply_chat_template(build_messages(text), tokenize=False, add_generation_prompt=True)
    p = tok(prompt, add_special_tokens=False)["input_ids"]
    a = tok(label_json(label) + tok.eos_token, add_special_tokens=False)["input_ids"]
    return mask_labels(p, a)
train_enc = [encode(*r) for r in train_rows]
val_enc = [encode(*r) for r in val_rows]
small_enc = [encode(*r) for r in small_rows]
lens = [len(e["input_ids"]) for e in train_enc]
print(f"tokens per training example: mean {np.mean(lens):.0f}, max {max(lens)}. Answer tokens: {sum(l != -100 for l in train_enc[0]['labels'])}")
lap("data ready")

## Step 1: the prompt-engineering baseline

Before you train anything, measure what prompting gives you. The base model gets the same system text as later. Zero-shot adds nothing. 4-shot adds one example per class. The examples come from the training pool, never from the test set. Decoding is greedy, with at most 16 new tokens.

In [ ]:
# 4. Baseline: base model in fp16, zero-shot and 4-shot, on the 300 test items
import gc
def free_gpu():
    gc.collect(); torch.cuda.empty_cache()

@torch.no_grad()
def hf_generate(model, messages_list, max_new_tokens=16, bs=16):
    tok.padding_side = "left"
    outs = []
    for i in range(0, len(messages_list), bs):
        texts = [tok.apply_chat_template(m, tokenize=False, add_generation_prompt=True) for m in messages_list[i:i + bs]]
        enc = tok(texts, return_tensors="pt", padding=True).to("cuda")
        gen = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False, use_cache=True, pad_token_id=tok.pad_token_id)
        outs += tok.batch_decode(gen[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
    return outs

test_texts, test_gold = [t for t, _ in test_rows], [y for _, y in test_rows]
base16 = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float16, device_map={"": 0}).eval()
for name, shots in [("zero-shot", ()), ("4-shot", SHOTS)]:
    t = time.time()
    outs = hf_generate(base16, [build_messages(x, shots) for x in test_texts])
    s = score(outs, test_gold)
    print(f"{name:10s} accuracy {s['accuracy']:.3f} | valid JSON {s['valid_json']:.3f} | {time.time()-t:.0f} s")
    print("   sample replies:", [o[:60] for o in outs[:3]])
    add("Prompting", f"base fp16, {name}", s["accuracy"], s["valid_json"], n=s["n"], note="prompt engineering, no training")
del base16; free_gpu()
lap("baseline done")

## Step 2: QLoRA SFT

What the next cell does:

1. It loads the base model in 4-bit NF4. Compute runs in fp16, because the T4 has no bf16.
2. It adds LoRA matrices (rank 16, alpha 32, dropout 0.05) to the attention and MLP projections. Only these train.
3. It trains with the plain `transformers` `Trainer`. A small collate function sets the loss to the answer tokens only. The prompt tokens get label -100.
4. It evaluates on the validation set every N steps. This gives the validation loss curve.

The learning rate is 2e-4. Effective batch size is 16. Gradient checkpointing trades extra compute for less memory.

In [ ]:
# 5. QLoRA training function, then the main run (2,000 examples, 2 epochs)
import matplotlib
import matplotlib.pyplot as plt

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
lora_cfg = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, task_type="CAUSAL_LM",
                      target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])

class Rows(torch.utils.data.Dataset):
    def __init__(self, rows): self.rows = rows
    def __len__(self): return len(self.rows)
    def __getitem__(self, i): return self.rows[i]
def collate(features):
    return {k: torch.tensor(v) for k, v in pad_batch(features, tok.pad_token_id).items()}

def make_args(out, **kw):
    a = dict(output_dir=out, per_device_eval_batch_size=16, learning_rate=2e-4, lr_scheduler_type="cosine",
             warmup_steps=5, fp16=True, gradient_checkpointing=True, gradient_checkpointing_kwargs={"use_reentrant": False},
             optim="adamw_torch", save_strategy="no", report_to="none", seed=SEED, remove_unused_columns=False,
             logging_first_step=True, eval_strategy="steps")
    a.update(kw)
    try:
        return TrainingArguments(**a)
    except TypeError:                       # older transformers call it evaluation_strategy
        a["evaluation_strategy"] = a.pop("eval_strategy")
        return TrainingArguments(**a)

def run_qlora(out, train_enc, epochs, bs, accum, eval_steps, log_steps):
    set_seed(SEED)
    m = AutoModelForCausalLM.from_pretrained(MODEL, quantization_config=bnb, torch_dtype=torch.float16, device_map={"": 0})
    m = prepare_model_for_kbit_training(m, gradient_checkpointing_kwargs={"use_reentrant": False})
    m = get_peft_model(m, lora_cfg)
    for p in m.parameters():                # fp16 Trainer needs fp32 trainable weights
        if p.requires_grad: p.data = p.data.float()
    try:
        trainable, total = m.get_nb_trainable_parameters()
        print(f"trainable parameters: {trainable:,} of {total:,} ({100*trainable/total:.2f} %)")
    except AttributeError:
        m.print_trainable_parameters()
    args = make_args(out, num_train_epochs=epochs, per_device_train_batch_size=bs, gradient_accumulation_steps=accum,
                     eval_steps=eval_steps, logging_steps=log_steps)
    trainer = Trainer(model=m, args=args, train_dataset=Rows(train_enc), eval_dataset=Rows(val_enc), data_collator=collate)
    torch.cuda.reset_peak_memory_stats()
    t = time.time()
    trainer.train()
    secs = time.time() - t
    peak = torch.cuda.max_memory_allocated() / 2**30
    print(f"training time {secs/60:.1f} min | peak GPU memory allocated {peak:.2f} GiB, reserved {torch.cuda.max_memory_reserved()/2**30:.2f} GiB")
    log = list(trainer.state.log_history)
    del trainer; free_gpu()
    return m, log, peak, secs

def plot_curves(ax, log, title):
    tr, ev = split_log(log)
    ax.plot(*zip(*tr), label="train loss")
    ax.plot(*zip(*ev), marker="o", label="validation loss")
    ax.set_xlabel("optimizer step"); ax.set_ylabel("loss on answer tokens"); ax.set_title(title); ax.legend(); ax.grid(alpha=.3)

model_news, log_news, peak_news, secs_news = run_qlora("/kaggle/working/run_news", train_enc, epochs=2, bs=8, accum=2, eval_steps=25, log_steps=10)
ADAPTER_DIR = "/kaggle/working/adapter_news"
model_news.save_pretrained(ADAPTER_DIR)
print("adapter saved:", ADAPTER_DIR, "|", sum(os.path.getsize(f"{ADAPTER_DIR}/{f}") for f in os.listdir(ADAPTER_DIR)) / 2**20, "MiB")
add("QLoRA SFT", "2,000 examples, 2 epochs", note=f"peak GPU memory {peak_news:.2f} GiB, train time {secs_news/60:.1f} min")
fig, ax = plt.subplots(figsize=(7, 4)); plot_curves(ax, log_news, "QLoRA: 2,000 examples, 2 epochs"); plt.show()
lap("main training done")

## Step 3: the overfitting run

Same recipe, same validation set. Only the training data changes: 64 examples, 8 epochs. The model sees each example many times. Watch the two curves. Training loss should keep falling. Validation loss tells you if the model still learns the task or only memorizes these 64 examples.

In [ ]:
# 6. Overfitting run: 64 examples, 8 epochs
model_small, log_small, peak_small, secs_small = run_qlora("/kaggle/working/run_overfit", small_enc, epochs=8, bs=8, accum=1, eval_steps=4, log_steps=2)
tr, ev = split_log(log_small)
best = min(ev, key=lambda x: x[1])
print(f"validation loss: lowest {best[1]:.3f} at step {best[0]}, last {ev[-1][1]:.3f} at step {ev[-1][0]}")
print(f"train loss: first {tr[0][1]:.3f}, last {tr[-1][1]:.3f}")
fig, axs = plt.subplots(1, 2, figsize=(13, 4))
plot_curves(axs[0], log_news, "QLoRA: 2,000 examples, 2 epochs")
plot_curves(axs[1], log_small, "Overfit run: 64 examples, 8 epochs (8 steps per epoch)")
plt.tight_layout(); plt.savefig("/kaggle/working/loss_curves.png", dpi=120); plt.show()
# Does the overfit adapter still classify? Test accuracy on the same 300 items.
model_small.eval()
s = score(hf_generate(model_small, [build_messages(x) for x in test_texts]), test_gold)
print(f"overfit adapter: test accuracy {s['accuracy']:.3f} | valid JSON {s['valid_json']:.3f}")
add("QLoRA SFT", "64 examples, 8 epochs (overfit run)", s["accuracy"], s["valid_json"], n=s["n"],
    note=f"lowest val loss {best[1]:.3f} at step {best[0]}, last {ev[-1][1]:.3f}")
del model_small; free_gpu()
lap("overfit run done")

## Step 4: test the fine-tuned model

Use the main adapter on the same 300 test items and the same prompt as the zero-shot baseline. The base is still 4-bit here. Later, vLLM serves the same adapter on an fp16 base. This is the normal path: train on a quantized base, serve on a full-precision base.

In [ ]:
# 7. Test accuracy and valid-JSON rate after fine-tuning
model_news.eval()
ft_outs = hf_generate(model_news, [build_messages(x) for x in test_texts])
s = score(ft_outs, test_gold)
print(f"QLoRA adapter: test accuracy {s['accuracy']:.3f} | valid JSON {s['valid_json']:.3f}")
print("sample replies:", [o[:60] for o in ft_outs[:3]])
add("QLoRA SFT", "HF, 4-bit base + adapter, 300 test items", s["accuracy"], s["valid_json"], n=s["n"])
pd.DataFrame([r for r in RESULTS if r["experiment"] in ("Prompting", "QLoRA SFT")])[["experiment", "setting", "accuracy", "valid_json"]]

## Step 5: catastrophic forgetting probe

The adapter trained on news labels only. Does it still do grade-school math? The probe uses 50 questions from the GSM8K test set. Both runs use the same 4-bit base, the same prompt and greedy decoding. One run uses the adapter. The other run turns the adapter off (`disable_adapter`). So the adapter is the only difference.

Limits: 50 questions is a small sample. One question is 2 points of accuracy. A difference of a few questions can be noise. Replies stop at 320 new tokens, so a long answer can be cut off. Also count how many replies start with `{`. That shows if the news format leaks into math answers.

In [ ]:
# 8. Forgetting probe: 50 GSM8K test questions, base vs base + adapter
try:
    gsm = load_dataset("openai/gsm8k", "main", split="test")
except Exception as e:
    print("openai/gsm8k failed:", repr(e)[:200], "-> trying gsm8k")
    gsm = load_dataset("gsm8k", "main", split="test")
pick = random.Random(SEED).sample(range(len(gsm)), 50)
qs = [gsm[i]["question"] for i in pick]
golds = [gsm8k_gold(gsm[i]["answer"]) for i in pick]
def math_msgs(q):
    return [{"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": q + "\nSolve it step by step. End with a final line: #### <number>"}]
def run_math(model):
    return hf_generate(model, [math_msgs(q) for q in qs], max_new_tokens=320, bs=25)
t = time.time()
with model_news.disable_adapter():
    outs_base = run_math(model_news)
outs_ad = run_math(model_news)
lap(f"forgetting probe generation took {time.time()-t:.0f} s")
for name, outs in [("base (adapter off)", outs_base), ("base + adapter", outs_ad)]:
    acc = sum(gsm8k_correct(o, g) for o, g in zip(outs, golds)) / len(golds)
    leak = sum(o.lstrip().startswith("{") for o in outs)
    print(f"{name:20s} GSM8K accuracy {acc:.2f} on {len(golds)} questions | replies starting with '{{': {leak}")
    add("Forgetting probe", f"GSM8K, {name}", acc, n=len(golds), note=f"{leak} replies start with '{{'. Small sample.")
print("\nadapter reply to question 1:", outs_ad[0][:200].replace("\n", " "))
print("base reply to question 1:   ", outs_base[0][:200].replace("\n", " "))

## Step 6: serve the adapter, then serve the merged model

Two ways to ship a LoRA fine-tune:

- **Adapter serving.** One vLLM server loads the base model once. It also loads one or more small adapters. Each request names a model: the base name or an adapter name (here `news`). Many adapters share one copy of the base weights. This saves memory when you have many tasks or many tenants. The cost is an extra low-rank computation per layer for requests that use an adapter.
- **Merging.** You add the LoRA change into the base weights and save a normal model. The server has no adapter and no extra step. But each merged model is a full copy of the weights. Ten tasks mean ten full models.

The vLLM flags: `--enable-lora` turns on adapter support. `--lora-modules news=<dir>` registers the adapter under the name `news`. `--max-lora-rank 16` must be at least the rank you trained with.

The merge uses an fp16 copy of the base. Do not merge into the 4-bit weights. The 4-bit weights are rounded, so the merged result would be rounded again.

The next cell frees the GPU in this kernel and merges. The servers run on GPU 1, so the kernel's leftover CUDA context on GPU 0 does not block them.

In [ ]:
# 9. Free GPU 0, merge the adapter into an fp16 base, save the merged model, define the server helpers
# Kaggle ships torchao 0.10, which this peft rejects when it loads an adapter onto an fp16 base.
# The 4-bit training path never reaches that check. Tell peft torchao is not available.
import peft.tuners.lora.torchao as _peft_torchao; _peft_torchao.is_torchao_available = lambda: False
del model_news; free_gpu()
MERGED_DIR = "/kaggle/working/merged_news"
base_fp16 = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float16, device_map={"": 0})
merged = PeftModel.from_pretrained(base_fp16, ADAPTER_DIR).merge_and_unload()
merged.save_pretrained(MERGED_DIR, safe_serialization=True)
tok.save_pretrained(MERGED_DIR)
del merged, base_fp16; free_gpu()
def dir_mib(d): return sum(os.path.getsize(f"{d}/{f}") for f in os.listdir(d)) / 2**20
print(f"adapter on disk: {dir_mib(ADAPTER_DIR):.1f} MiB | merged model on disk: {dir_mib(MERGED_DIR):.0f} MiB")

BASE_URL = "http://localhost:8000"
MAX_LEN, GPU_UTIL = 2048, 0.80
def gpu_used_mib():
    out = subprocess.check_output(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"], text=True)
    return [int(x) for x in out.split()]
SERVE_GPU = 1 if len(gpu_used_mib()) > 1 else 0     # one-GPU fallback: serve on GPU 0 after the kernel freed it
print("vLLM will use GPU", SERVE_GPU, "| memory in use per GPU (MiB):", gpu_used_mib())

def start_server(name, model, extra=()):
    log = f"/kaggle/working/vllm_{name}.log"
    cmd = ["vllm", "serve", model, "--dtype", "half", "--max-model-len", str(MAX_LEN),
           "--gpu-memory-utilization", str(GPU_UTIL), "--port", "8000", *extra]
    print(name, "->", " ".join(cmd[1:]))
    srv = subprocess.Popen(cmd, stdout=open(log, "w"), stderr=subprocess.STDOUT,
                           env=dict(os.environ, CUDA_VISIBLE_DEVICES=str(SERVE_GPU)))
    srv.log = log
    t0 = time.time()
    while True:
        try:
            urllib.request.urlopen(f"{BASE_URL}/health"); break
        except Exception:
            if srv.poll() is not None:
                print(open(log).read()[-4000:]); raise RuntimeError("server exited, see log tail above")
            if time.time() - t0 > 1200:
                srv.kill(); raise RuntimeError("server not ready after 20 min")
            time.sleep(2)
    print(f"{name}: server ready after {time.time()-t0:.0f} s")
    return srv

def stop_server(srv):
    srv.terminate()
    try: srv.wait(timeout=90)
    except subprocess.TimeoutExpired: srv.kill(); srv.wait()
    for _ in range(60):                     # wait until the GPU memory is free again
        if gpu_used_mib()[SERVE_GPU] < 1500: return
        time.sleep(2)
    print("warning: GPU memory not free after 120 s:", gpu_used_mib())

def chat(model, messages, max_tokens=16):
    body = json.dumps({"model": model, "messages": messages, "temperature": 0, "max_tokens": max_tokens}).encode()
    req = urllib.request.Request(f"{BASE_URL}/v1/chat/completions", body, {"Content-Type": "application/json"})
    t = time.perf_counter()
    r = json.load(urllib.request.urlopen(req, timeout=120))
    return r["choices"][0]["message"]["content"], (time.perf_counter() - t) * 1000

# 100 balanced test items, sent one at a time. Latency here is single-request latency, not throughput.
(sel,) = balanced_split(test_gold, [100], SEED)
sel_texts, sel_gold = [test_texts[i] for i in sel], [test_gold[i] for i in sel]
def eval_served(model_name):
    for t in sel_texts[:3]:                 # warm-up. The first adapter request also loads the adapter.
        chat(model_name, build_messages(t))
    res = [chat(model_name, build_messages(t)) for t in sel_texts]
    s = score([r[0] for r in res], sel_gold)
    return s, statistics.median(r[1] for r in res)
s_hf = score([ft_outs[i] for i in sel], sel_gold)
add("Serving", "HF, 4-bit base + adapter, same 100 items", s_hf["accuracy"], s_hf["valid_json"], n=100)
lap("merged model saved")

In [ ]:
# 10. Serve the base model with the adapter. Query `base` and `news` through /v1/chat/completions.
srv = start_server("lora", MODEL, ["--enable-lora", "--lora-modules", f"news={ADAPTER_DIR}", "--max-lora-rank", "16"])
try:
    print("served models:", [m["id"] for m in json.load(urllib.request.urlopen(f"{BASE_URL}/v1/models"))["data"]])
    for served, label in [(MODEL, "vLLM fp16 base, no adapter"), ("news", "vLLM fp16 base + adapter `news`")]:
        s, lat = eval_served(served)
        print(f"{label:34s} accuracy {s['accuracy']:.3f} | valid JSON {s['valid_json']:.3f} | median latency {lat:.1f} ms")
        add("Serving", label, s["accuracy"], s["valid_json"], lat, 100)
finally:
    stop_server(srv)
!grep -iE "lora" /kaggle/working/vllm_lora.log | head -5 | cut -c1-200
lap("adapter serving done")

In [ ]:
# 11. Serve the merged model. No LoRA flags. Same 100 items.
srv = start_server("merged", MERGED_DIR, ["--served-model-name", "merged"])
try:
    s, lat = eval_served("merged")
    print(f"{'vLLM merged fp16 model':34s} accuracy {s['accuracy']:.3f} | valid JSON {s['valid_json']:.3f} | median latency {lat:.1f} ms")
    add("Serving", "vLLM merged fp16 model", s["accuracy"], s["valid_json"], lat, 100)
finally:
    stop_server(srv)
lap("merged serving done")

In [ ]:
# 12. Summary table
df = pd.DataFrame(RESULTS)
for c in ("accuracy", "valid_json"): df[c] = df[c].astype(float).round(3)
df["median_latency_ms"] = df["median_latency_ms"].astype(float).round(1)
df.to_csv("/kaggle/working/finetune_results.csv", index=False)
print(df.to_string(index=False))
print(f"\ntotal wall time {(time.time()-T0)/60:.1f} min | saved /kaggle/working/finetune_results.csv and loss_curves.png")

## What to look for

- **Prompting vs fine-tuning.** Compare accuracy and valid-JSON rate of the prompt rows with the QLoRA row. A prompt can show the format. Fine-tuning fixes it in the weights, so the prompt stays short.
- **Overfit run.** Find the step where validation loss is lowest. After that step, more training only fits the 64 examples. Check the test accuracy of that adapter too. Loss and accuracy do not always move together.
- **Forgetting probe.** Compare the two GSM8K scores and the count of replies that start with `{`. The sample has 50 questions, so treat small gaps as noise. Do not conclude from this probe alone that forgetting is absent.
- **Serving.** `base` and `news` come from the same loaded weights. The adapter request does extra work per layer, so its latency should be equal or higher. The merged model should match the adapter in accuracy. The merged model has no extra step. The HF row and the vLLM adapter row use the same adapter but different base precision (4-bit vs fp16). Small differences are expected.
- Quantization is a second change in this experiment. The baseline rows use an fp16 base. The HF fine-tuned row uses a 4-bit base. The forgetting probe avoids this mix, because both runs share the 4-bit base.

## Exam rules this proves

1. **customization.1**: Adapter serving keeps one copy of the base weights and swaps small adapters per request. Merging gives one standalone model with no adapter step, but one full copy per task.
2. **customization.3**: SFT trains on input and output pairs. PEFT trains few weights. LoRA learns small low-rank matrices next to frozen weights. QLoRA does the same on a 4-bit base so the job fits a small GPU.
3. **customization.4**: Overfitting shows as falling train loss with rising validation loss. Catastrophic forgetting shows as worse scores on tasks outside the tuning data. Always measure both on held-out data.
4. **customization.5, customization.6**: Fine-tune when prompting cannot give a reliable format or behavior and you have enough clean labelled examples. Balance the classes, hold out validation and test sets, and keep test data out of the prompt examples.
5. **customization.9**: Try prompting first, because it is the cheapest. Use RAG to give the model fresh or private facts. Use fine-tuning to change format, style, or behavior.